### Import Dependencies

In [ ]:
from pydantic import BaseModel

from langsmith import traceable, get_current_run_tree

import instructor

from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Send

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, AnyMessage
from IPython.display import Image, display

from typing import Literal, Dict, Any, Annotated, List
from operator import add

import random
import openai
import pandas as pd

from jinja2 import Template

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import (
    VectorParams,
    Distance,
    SparseVectorParams,
    Modifier,
    PayloadSchemaType,
    PointStruct,
    Document,
    Prefetch,
    FusionQuery,
)


In [ ]:
from dotenv import load_dotenv

load_dotenv("../../.env")

In [ ]:
from typing import TypedDict
from api.agents.utils.prompt_management import prompt_template_config
import cohere
from pydantic import Field


MODEL_NAME_AGENT = "gpt-5.4-mini"
PROVIDER_NAME_AGENT = "openai"


MODEL_NAME_EMBEDDING = "text-embedding-3-small"


class RAGUsedContextSimple(BaseModel):
    id: str = Field(description="ID of the item used to answer the question")
    description: str = Field(
        description="Description of the item corresponding to the id"
    )


class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")
    references: list[RAGUsedContextSimple] = Field(
        description="List of items used to answer the question"
    )


def get_embedding(text, model=MODEL_NAME_EMBEDDING):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding

### Agent Graph with Loopback from Tools (ReAct Agent)

In [ ]:
embedding_model = "text-embedding-3-small"

@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={"ls_provider": "openai", "ls_model_name": embedding_model},
)
def get_embedding(text, model=embedding_model):
    response = openai.embeddings.create(input=text, model=model)
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens,
        }
    return response.data[0].embedding


RetrievedData = TypedDict(
    "RetrievedData",
    {
        "retrieved_context_ids": list[str],
        "retrieved_context": list[str],
        "similarity_scores": list[float],
        "retrieved_context_ratings": list[float],
    },
)

HYBRID_SEARCH_COLLECTION_NAME = "Amazon-items-collection-01-hybrid-search"


@traceable(name="retrieve_data", run_type="retriever")
def retrieve_data(
    query, qdrant_client: QdrantClient, k=5, hybrid=True
) -> RetrievedData:
    query_embedding = get_embedding(query)
    if hybrid:
        results = qdrant_client.query_points(
            collection_name=HYBRID_SEARCH_COLLECTION_NAME,
            prefetch=[
                Prefetch(
                    query=query_embedding,
                    using="text-embedding-3-small",
                    limit=20,
                ),
                Prefetch(
                    query=Document(
                        text=query,
                        model="qdrant/bm25",
                    ),
                    using="bm25",
                    limit=20,
                ),
            ],
            query=models.RrfQuery(rrf=models.Rrf(weights=[3, 1])),
            limit=k,
        )
    else:
        results = qdrant_client.query_points(
            collection_name=HYBRID_SEARCH_COLLECTION_NAME,
            query=query_embedding,
            using="text-embedding-3-small",
            limit=k,
        )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        if not result.payload:
            raise ValueError("No payload found in Qdrant ScoredPoint")
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
    }


RERANK_MODEL = "rerank-v4.0-pro"


@traceable(
    name="rerank_data",
    run_type="chain",
    metadata={"ls_provider": "cohere", "ls_model_name": RERANK_MODEL},
)
def rerank_data(query: str, context: RetrievedData, top_k=5) -> RetrievedData:
    cohere_client = cohere.ClientV2()
    response = cohere_client.rerank(
        model=RERANK_MODEL,
        query=query,
        documents=context["retrieved_context"],
        top_n=top_k,
    )
    order = [result.index for result in response.results]
    return {
        "retrieved_context_ids": [context["retrieved_context_ids"][i] for i in order],
        "retrieved_context": [context["retrieved_context"][i] for i in order],
        "similarity_scores": [context["similarity_scores"][i] for i in order],
        "retrieved_context_ratings": [
            context["retrieved_context_ratings"][i] for i in order
        ],
    }


@traceable(name="format_retrieved_context", run_type="prompt")
def process_context(context: RetrievedData) -> str:
    formatted_context = ""

    for id, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_context"],
        context["retrieved_context_ratings"],
    ):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

@tool
def get_formatted_item_context(query: str, top_k: int = 5) -> str:
    """Get the top k context, each representing an inventory item for a given query.
    
    Args:
        query: The query to get the top k context for
        top_k: The number of context chunks to retrieve, works best with 5 or more
    
    Returns:
        A string of the top k context chunks with IDs and average ratings prepending each chunk, each representing an inventory item for a given query.
    """

    qdrant_client = QdrantClient(url="http://localhost:6333")
    retrieved_context = retrieve_data(
        query, qdrant_client, k=20, hybrid=True
    )

    retrieved_context = rerank_data(query, retrieved_context, top_k=top_k)
    formatted_context = process_context(retrieved_context)
    return formatted_context


### State and Pydantic Models for Structured Outputs

In [ ]:
from enum import StrEnum

class FinalResponse(BaseModel):
  answer: str = Field(description="Answer to the question")
  references: list[RAGUsedContextSimple] = Field(description="List of items used to answer the question")


class StateUpdate(TypedDict, total=False):
  messages: Annotated[List[AnyMessage], add]
  question_relevant: bool
  iteration: int
  answer: str
  final_answer: bool
  references: list[RAGUsedContextSimple]

class State(BaseModel):
    messages: Annotated[List[AnyMessage], add] = []
    question_relevant: bool = False
    iteration: int = 0
    answer: str = ""
    final_answer: bool = False
    references: list[RAGUsedContextSimple] = []

assert set(StateUpdate.__annotations__) == set(State.model_fields)

In [ ]:
from langchain_openai import ChatOpenAI

@traceable(
    name="agent_node",
    run_type="llm",
    metadata={
        "ls_provider": PROVIDER_NAME_AGENT,
        "ls_model_name": MODEL_NAME_AGENT,
    }
)
def agent_node(state: State) -> StateUpdate:

    prompt_template = """You are a shopping assistant that answers customer questions about products currently in stock.

## Instructions

- Use the available tools to answer product questions. Do not fabricate product details.
- When a question involves multiple products or sub-questions, issue all tool calls at once. Never repeat a tool call you already made.
- When describing products, include detailed specifications in bullet points.
- If tools return no relevant results, tell the customer and ask them to refine their query.
- Only answer questions about products in stock. If a question is unrelated, ask the customer to clarify what product they're interested in.
- In references, include every chunk that contributed to your answer with the chunk id and product name.
- Refer to retrieved data as "available products", never as "context".
- Try answering queries that are not precise, if specific names or brands are missing apply broad searches.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini", reasoning_effort="none", use_responses_api=True
    )
    llm_with_tools = llm.bind_tools([get_formatted_item_context, FinalResponse], tool_choice="any")

    final_answer = False
    answer = ""
    references: list[RAGUsedContextSimple] = []
    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    if len(response.tool_calls) > 0:
        for tool_call in response.tool_calls:
            if tool_call.get("name") == FinalResponse.__name__:
                final_answer = True
                final_response_validated = FinalResponse.model_validate(tool_call.get("args"))
                references.extend(final_response_validated.references)
                answer = final_response_validated.answer
                break


    return {"messages": [response],
            "final_answer": final_answer,
            "iteration": state.iteration + 1,
            "answer": answer,
            "references": references}


In [ ]:
class Nodes(StrEnum):
  AGENT = "agent"
  INTENT_ROUTER = "intent_router"
  TOOLS = "tools"
  END = END

def tool_router(state: State) -> Nodes:
  if state.final_answer:
    return Nodes.END
  if state.iteration > 2:
    return Nodes.END

  last_message = state.messages[-1]
  if isinstance(last_message, AIMessage) and len(last_message.tool_calls) > 0:
    return Nodes.TOOLS
  else:
    return Nodes.AGENT
    

### User Intent Router Node

In [ ]:
class IntentRouterResponse(BaseModel):
    question_relevant: bool
    answer: str = Field(description="A clarifying question, if the user's initial query is not relevant.")

In [ ]:

from langchain_core.messages import convert_to_openai_messages


@traceable(
  name="route_intent",
  run_type="llm",
  metadata={
    "ls_provider": PROVIDER_NAME_AGENT,
    "ls_model_name": MODEL_NAME_AGENT,
  }
)
def intent_router_node(state: State) -> IntentRouterResponse:

    prompt_template = """You are a relevance router for a shopping assistant that answers questions about products in stock.

## Instructions

- Determine whether the question is about products, inventory, or purchasing.
- Questions about product features, availability, pricing, comparisons, and recommendations are relevant.
- Questions about store policies, personal advice, or unrelated topics are not relevant.

## Examples

Question: "Do you have running shoes under $100?"
Relevant: yes

Question: "What's the weather like today?"
Relevant: no - not related to products

Question: "Can you help me write an essay?"
Relevant: no - not related to products

Question: "Which laptop has the best battery life?"
Relevant: yes

Question: "What's your return policy?"
Relevant: no - about store policy, not product information
"""

    template = Template(prompt_template)

    prompt = template.render()

    messages = state.messages

    conversation = []

    for message in messages:
      conversation.append(convert_to_openai_messages(message))

    client = instructor.from_provider(
        PROVIDER_NAME_AGENT + "/" + MODEL_NAME_AGENT, mode=instructor.Mode.RESPONSES_TOOLS
    )

    response, raw_response = client.create_with_completion(
        messages=[{"role": "system", "content": prompt}, *conversation], 
        reasoning={"effort": "none"},
        response_model=IntentRouterResponse,
    )

    return response

In [ ]:
def intent_router_conditional_edges(state: State) -> Nodes:
  if state.question_relevant:
    return Nodes.AGENT
  else:
    return Nodes.END


In [ ]:
workflow = StateGraph(State)
tools = [get_formatted_item_context]
tool_node = ToolNode(tools)

workflow.add_node(Nodes.TOOLS, tool_node)
workflow.add_node(Nodes.AGENT, agent_node)
workflow.add_node(Nodes.INTENT_ROUTER, intent_router_node)

workflow.add_edge(START, Nodes.INTENT_ROUTER)


workflow.add_conditional_edges(
    Nodes.INTENT_ROUTER, intent_router_conditional_edges,
    {
        Nodes.AGENT: Nodes.AGENT,
        Nodes.END: Nodes.END,
    }
)

workflow.add_conditional_edges(
    Nodes.AGENT, 
    tool_router, 
    {
        Nodes.TOOLS: Nodes.TOOLS,
        Nodes.END: END
    }
)

workflow.add_edge(Nodes.TOOLS, Nodes.AGENT)

workflow.add_edge(Nodes.AGENT, END)

graph = workflow.compile()


In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:


initial_state = State(
    messages=[
        HumanMessage(
            content="Can I get a tablet for my kid, a watch for me, a laptop for my wife, and a waterproof speaker?"
        )
    ]
)

In [ ]:
result = graph.invoke(initial_state)

In [ ]:
result

In [ ]:
print(result["answer"])

In [ ]:
initial_state = State(
    messages=[
        HumanMessage(
            content="What is the meaning of life?"
            
        )
    ]
)


In [ ]:
result = graph.invoke(initial_state)

In [ ]:
result